# 06 Binoculars

Binoculars is zero-shot, so there is nothing to train. `scripts/run_binoculars.py`
scored every calibration and test text once on Viking, and this notebook turns
those scores into results.

That makes it much shorter than notebook 05. There are no training sets and no
seeds, so there is one threshold and one row per family rather than forty runs.
Binoculars is also unaffected by generator shift by construction, since no family
can be withheld from training it never did, which is what makes it the reference
line the trained detectors are measured against (section 3.5).

Because Binoculars performs poorly on MAGE, `scripts/validate_binoculars_hc3.py`
checks the implementation on HC3, where the machine text is ChatGPT. That run and
section 6 together establish that the MAGE result is a property of MAGE's
generators rather than of the code.

## 1. Load the corpus and the scores

The scores file has one row per scored text, with `text_id` to join on, `part` to
say whether the text came from the calibration or the test split, and `score`.

The scores were sign-flipped when they were saved, so a higher score means more
likely machine-generated, matching every other detector in the study. A raw
Binoculars score runs the other way.

In [1]:
import numpy as np
import pandas as pd

from logos.paths import DATA_DIR
from logos.evaluation.folds import calibration_set, test_set
from logos.evaluation.metrics import (
    calibrate_threshold, tpr_at_threshold, fpr_at_threshold, auroc
)

corpus = pd.read_parquet(DATA_DIR / "mage" / "corpus.parquet")
scores = pd.read_parquet(DATA_DIR / "mage" / "scores" / "binoculars" / "all.parquet")

print(f"corpus {len(corpus)} rows, scores {len(scores)} rows")
scores.head()

corpus 392385 rows, scores 80882 rows


,text_id,part,score
0,287357,calibration,-1.022346
1,287358,calibration,-0.995238
2,287359,calibration,-0.978836
3,287360,calibration,-0.947619
4,287361,calibration,-1.000000


## 2. Coverage and direction checks

Two things to confirm before reading anything into the numbers.

First, every calibration and test text should pick up exactly one score. If the
counts do not match, the join has either lost texts or duplicated them.

Second, machine texts should score higher than human texts on average. If they do
not, the sign flip went astray and everything below is meaningless.

The check is on the whole test set, so passing it does not mean it passes family
by family. It does not, and section 5 shows three families where the ordering
reverses. The margin is also slim, 0.033, against 0.330 between the two classes
on HC3.

In [2]:
calibration = calibration_set(corpus).merge(
    scores[scores["part"] == "calibration"], on="text_id"
)
test = test_set(corpus).merge(scores[scores["part"] == "test"], on="text_id")

print(f"calibration {len(calibration)} of {len(calibration_set(corpus))}")
print(f"test        {len(test)} of {len(test_set(corpus))}")

print(f"\nmean score, human test texts   {test[test['label'] == 0]['score'].mean():.4f}")
print(f"mean score, machine test texts {test[test['label'] == 1]['score'].mean():.4f}")

calibration 28407 of 28407
test        52475 of 52475

mean score, human test texts   -0.9708
mean score, machine test texts -0.9381


## 3. Calibrate the threshold

One threshold for everything. The trained detectors need a threshold per training
set, because each training set produces a different model and therefore a
different spread of scores. Binoculars produces the same scores whatever is
withheld, so it has a single threshold.

The calibration texts are the human texts from MAGE's validation split, which are
available at deployment in a way text from an unseen generator is not.

τ* comes out at −0.734. Undoing the sign flip, that is a raw Binoculars cut-off of
0.734, below the 0.8536 the authors published for their own data. That is the
opposite of what the error rates imply, since this study allows 1% of human texts
to be flagged against their 0.01%, and a looser rate should push the cut-off up
rather than down. The HC3 validation run
(`scripts/validate_binoculars_hc3.py`) gives −0.739 on a different corpus
entirely, so this reflects the human text in both corpora used here rather than
anything particular to MAGE.

In [3]:
threshold = calibrate_threshold(
    calibration["score"].to_numpy(), calibration["label"].to_numpy()
)
print(f"tau* = {threshold:.6f}")

tau* = -0.734043


## 4. Per-family results

Each family is tested on the shared human test texts plus that family's machine
texts, exactly as in notebook 05. TPR is the share of that family's machine texts
caught at the frozen threshold, and AUROC is computed on the same set as a
secondary measure for RQ1.

`n_machine` is kept because the size of a family's test set sets how precise its
TPR can be. GLM-130B is the smallest at 919 texts.

In [4]:
rows = []
for family in sorted(corpus[corpus["label"] == 1]["family"].dropna().unique()):
    subset = test_set(corpus, target=family).merge(
        scores[scores["part"] == "test"], on="text_id"
    )
    family_scores = subset["score"].to_numpy()
    family_labels = subset["label"].to_numpy()

    rows.append({
        "detector": "binoculars",
        "family": family,
        "condition": "Zero-shot",
        "tpr": tpr_at_threshold(family_scores, family_labels, threshold),
        "auroc": auroc(family_scores, family_labels),
        "test_fpr": fpr_at_threshold(family_scores, family_labels, threshold),
        "n_machine": int((family_labels == 1).sum()),
    })

results = pd.DataFrame(rows).round(4)
results

,detector,family,condition,tpr,auroc,test_fpr,n_machine
0,binoculars,BigScience,Zero-shot,0.0178,0.4837,0.0107,2692
1,binoculars,EleutherAI,Zero-shot,0.0007,0.1468,0.0107,1442
2,binoculars,FLAN-T5,Zero-shot,0.0163,0.5268,0.0107,4658
3,binoculars,GLM-130B,Zero-shot,0.3090,0.9458,0.0107,919
4,binoculars,LLaMA,Zero-shot,0.1291,0.9453,0.0107,3710
5,binoculars,OPT,Zero-shot,0.0004,0.3273,0.0107,8012
6,binoculars,OpenAI GPT,Zero-shot,0.2918,0.9266,0.0107,2790


### Three families are inverted

GLM-130B, LLaMA and OpenAI GPT are detected well, at AUROC 0.93 to 0.95. FLAN-T5
at 0.527 is near guessing. The other three fall below 0.5, with BigScience at
0.484, OPT at 0.327 and EleutherAI at 0.147.

Below 0.5 means the detector is reliably wrong rather than merely uninformative,
because it rates that family's machine text as more human-looking than genuine
human text. Section 6 works through the alternative explanations. The HC3
validation run rules out the implementation, reaching TPR@1%FPR of 0.896 and
AUROC of 0.996 on ChatGPT text using this same code.

What is left is the generators. Binoculars assumes machine text is unusually
predictable. The models behind the three inverted families, which are GPT-J and
GPT-NeoX, the OPT series, and BLOOM and T0, were released in 2021 and 2022, and
their output is often repetitive, truncated or incoherent, which Falcon finds
hard to predict. Binoculars reads that unpredictability as a human signature.

## 5. Is it length, the data, or the generators?

Test texts keep their natural lengths, and §3.2 records human and machine medians
differing by up to 3.4 times within a domain. Binoculars scores are known to be
length-sensitive, so length is the first alternative explanation to rule out.

In [5]:
test["words"] = test["text"].str.split().str.len()
print("correlation between score and length:",
      round(test["score"].corr(test["words"]), 3))

human_median = test[test["label"] == 0]["words"].median()
print(f"\nhuman test texts, median words: {human_median:.0f}\n")

for family in sorted(test[test["label"] == 1]["family"].dropna().unique()):
    machine = test[test["family"] == family]
    print(f"{family:12s} median words {machine['words'].median():6.0f}  "
          f"ratio to human {machine['words'].median() / human_median:5.2f}  "
          f"mean score {machine['score'].mean():7.4f}")

correlation between score and length: 0.027

human test texts, median words: 116

BigScience   median words     50  ratio to human  0.43  mean score -0.9711
EleutherAI   median words    164  ratio to human  1.41  mean score -1.0896
FLAN-T5      median words     41  ratio to human  0.35  mean score -0.9676
GLM-130B     median words    595  ratio to human  5.13  mean score -0.7728
LLaMA        median words    417  ratio to human  3.59  mean score -0.7917
OPT          median words    145  ratio to human  1.25  mean score -1.0226
OpenAI GPT   median words    109  ratio to human  0.94  mean score -0.7854


The correlation is 0.027, effectively nothing. More telling is OpenAI GPT, whose
median of 109 words is closest to the human median of 116 and which is among the
best detected at AUROC 0.927, while EleutherAI at 1.41 times human length and OPT
at 1.25 are the inverted ones. Among the three families closest in length to each
other, at 109, 145 and 164 words, mean scores are −0.785, −1.023 and −1.090. The
score separates by family, not by length.

The texts Binoculars is most confident are human show why.

In [6]:
most_human = test[(test["family"] == "EleutherAI") & (test["label"] == 1)].nsmallest(3, "score")
for _, row in most_human.iterrows():
    print(f"--- score {row['score']:.4f}, {row['words']} words, {row['domain']}")
    print(row["text"][:400], "\n")

--- score -1.4061, 121 words, xsum
The club's latest accounts, for the year ending May 2014, also show the Bluebirds owe around PS174m - most of it to owner Vincent Tan. This was despite revenue increasing slightly from 89M in 2013-14 season and net debt dropping by just over 50 (24) during that time period too (to a new low)."What we can say is this has been part one," said Chandler soon after taking his place at Cardiff City Stad 

--- score -1.3978, 64 words, sci_gen
This paper investigates a general framework to discover categories of unlabeled scene images according to their appearances (i.e., textures and structures). We jointly solve the two coupled tasks in an end-toend manner: category discovery from appearance information, as well visual object recognition within each discovered class using conventional classification methods such that accuracy on both  

--- score -1.3762, 110 words, hswag
show the kind of fish they have found as they lay it out on the frozen ground. The ice

These are visibly degraded, with garbled figures, unmatched quotation marks,
sentences cut off mid-word and passages that make no sense. It is not only the
extreme tail either, since EleutherAI's whole distribution is shifted, mean
−1.090 against humans' −0.971.

One detail in the first of those texts needs ruling out. "PS174m" is a pound sign
mangled by a character-encoding error, which is a collection artefact rather than
anything a generator produced. If that kind of damage were concentrated in machine
text, it would be doing some of the work attributed to the generators above.

In [7]:
import re
from collections import Counter

pound = corpus["text"].str.contains(r"PS\d")

print(f"xsum is {(corpus['domain'] == 'xsum').mean():.2%} of the corpus\n")
print("rate of mangled pound signs, by class")
print(pound.groupby(corpus["label"]).mean().round(4))
print("\nby domain and class")
print(pound.groupby([corpus["domain"], corpus["label"]]).mean().unstack().round(4))

# Does the lexical baseline's minimum document frequency of 5 let any through?
document_frequency = Counter()
for text in corpus["text"]:
    document_frequency.update(set(re.findall(r"\bps\d\w*", text.lower())))

surviving = {token: n for token, n in document_frequency.items() if n >= 5}
print(f"\n{len(document_frequency)} distinct artefact tokens, "
      f"{len(surviving)} appear in 5 or more texts")
print(sorted(surviving.items(), key=lambda pair: -pair[1])[:15])

xsum is 9.64% of the corpus

rate of mangled pound signs, by class
label
0    0.0163
1    0.0059
Name: text, dtype: float64

by domain and class
label         0       1
domain                 
cmv      0.0024  0.0007
eli5     0.0037  0.0012
hswag    0.0000  0.0002
roct     0.0000  0.0002
sci_gen  0.0000  0.0005
squad    0.0022  0.0037
tldr     0.0008  0.0007
wp       0.0015  0.0013
xsum     0.1931  0.0463
yelp     0.0023  0.0002

815 distinct artefact tokens, 206 appear in 5 or more texts
[('ps1', 452), ('ps2', 352), ('ps3', 293), ('ps4', 219), ('ps6', 138), ('ps5', 126), ('ps1bn', 106), ('ps100', 105), ('ps20', 94), ('ps10', 94), ('ps1m', 85), ('ps7', 81), ('ps9', 70), ('ps11', 69), ('ps20m', 68)]


The artefact is real but it runs the other way. It affects 1.6% of human texts
against 0.6% of machine texts, so it makes human writing slightly harder to
predict rather than machine writing, which is the direction Binoculars is
supposed to go for human text. It cannot therefore be what inverts the machine
side.

It is also confined to xsum, which is 9.64% of the corpus, and the tokens it
leaves cannot be separated from genuine references to games consoles, since `ps1`
through `ps5` are PlayStation models as often as pound amounts. It is left in
place and recorded in §3.6.

So length is ruled out, the encoding artefact is ruled out, and the HC3 run rules
out the implementation. The generators are what remain.

## 6. Summary and save

The saved file uses the same columns as `summary_lexical.csv`, so the two can be
stacked when Chapter 4's tables are built.

The standard deviation columns are left empty rather than set to zero. Zero would
say the seeds were run and made no difference; empty says there were no seeds to
run, which is the accurate statement for a zero-shot detector.

There are seven rows rather than fourteen, because there is only one condition.
When the comparison table is built, each Binoculars figure is repeated across the
in-distribution and LOFO columns, with the caption noting the two are identical by
construction.

In [8]:
print(f"mean TPR across families  {results['tpr'].mean():.4f}")
worst = results.loc[results["tpr"].idxmin()]
print(f"worst-case TPR            {worst['tpr']:.4f} ({worst['family']}, "
      f"{round(worst['tpr'] * worst['n_machine'])} of {worst['n_machine']} texts)")

summary = pd.DataFrame({
    "detector": results["detector"],
    "family": results["family"],
    "condition": results["condition"],
    "tpr_mean": results["tpr"],
    "tpr_std": np.nan,
    "auroc_mean": results["auroc"],
    "auroc_std": np.nan,
    "test_fpr_mean": results["test_fpr"],
    "test_fpr_std": np.nan,
})

out_path = DATA_DIR / "results" / "summary_binoculars.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
summary.to_csv(out_path, index=False)

print(f"\nsaved to {out_path}")
summary

mean TPR across families  0.1093
worst-case TPR            0.0004 (OPT, 3 of 8012 texts)

saved to /Users/xwm517/Desktop/logos/data/results/summary_binoculars.csv


,detector,family,condition,tpr_mean,tpr_std,auroc_mean,auroc_std,test_fpr_mean,test_fpr_std
0,binoculars,BigScience,Zero-shot,0.0178,NaN,0.4837,NaN,0.0107,NaN
1,binoculars,EleutherAI,Zero-shot,0.0007,NaN,0.1468,NaN,0.0107,NaN
2,binoculars,FLAN-T5,Zero-shot,0.0163,NaN,0.5268,NaN,0.0107,NaN
3,binoculars,GLM-130B,Zero-shot,0.3090,NaN,0.9458,NaN,0.0107,NaN
4,binoculars,LLaMA,Zero-shot,0.1291,NaN,0.9453,NaN,0.0107,NaN
5,binoculars,OPT,Zero-shot,0.0004,NaN,0.3273,NaN,0.0107,NaN
6,binoculars,OpenAI GPT,Zero-shot,0.2918,NaN,0.9266,NaN,0.0107,NaN


In [10]:
for window in [25, 50, 100]:
    share = (test["words"] < window).mean()
    print(f"window {window:3d}: {share:.1%} of test texts fall back")

print()
print(test.groupby("family")["words"].apply(lambda w: (w < 25).mean()).round(3))
print(test.groupby("label")["words"].apply(lambda w: (w < 25).mean()).round(4))

window  25: 3.1% of test texts fall back
window  50: 24.4% of test texts fall back
window 100: 44.9% of test texts fall back

family
BigScience    0.012
EleutherAI    0.001
FLAN-T5       0.018
GLM-130B      0.017
LLaMA         0.001
OPT           0.002
OpenAI GPT    0.005
Name: words, dtype: float64
label
0    0.0507
1    0.0070
Name: words, dtype: float64
